# Train EfficientNet-B3 Classifier — PHIÊN BẢN BLACKOUT (xoá nền ngoài mask phổi)

Chest X-ray — COVID-19 / Pneumonia / Normal

**Khác biệt DUY NHẤT so với `train_classifier_cropped.ipynb`:** dataset dùng thêm
`blackout=True` (xem `src/dataset.py::crop_to_lung_bbox_blackout`) — ngoài việc crop
theo bounding box mask phổi như bản trước, mọi pixel NẰM TRONG bounding box nhưng
NGOÀI hình dạng phổi thật (watermark, logo, thiết bị y tế...) còn bị xoá về 0 (đen).
Đây là bước tiếp theo sau khi phát hiện: crop bounding box KHÔNG loại bỏ được vật thể
nằm trong box nhưng ngoài phổi — xem `docs/BAO_CAO_KET_QUA_HUAN_LUYEN.md` Phần 5.4
(ca cụ thể `sample_covid.png`: containment=0.499 dù đã train trên ảnh crop). Mọi
hyperparameter khác GIỮ NGUYÊN so với 2 bản trước để so sánh công bằng (ablation study
— chỉ đổi 1 biến so với bản crop).


In [ ]:
import os
import sys
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    """Tim thu muc goc repo (chua src/) bang cach di nguoc len tu thu muc hien tai —
    can thiet vi kernel Jupyter co the khoi dong o notebooks/ thay vi goc repo."""
    for candidate in [start, *start.parents]:
        if (candidate / "src").is_dir():
            return candidate
    raise RuntimeError(
        "Khong tim thay thu muc src/ tu day - hay mo notebook nay tu trong repo "
        "Chest-X-ray-Segmentation-and-Diagnosis-of-Pneumonia-and-COVID-19."
    )

REPO_ROOT = _find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Working directory:", Path.cwd())


In [ ]:
# Sanity check GPU — chay truoc khi train
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)

In [ ]:
# Set seed — dam bao reproducibility
import random, os
import numpy as np
import torch

def set_seed(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
import numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import f1_score
from tqdm import tqdm
from pathlib import Path

from src.dataset import (
    ChestXrayClassificationDataset,
    get_train_transforms, get_val_transforms,
    NUM_CLASSES, IDX_TO_CLASS,
)
from src.model import (
    build_classifier, freeze_backbone,
    unfreeze_last_blocks, unfreeze_all,
    count_trainable_params,
)

In [ ]:
# CONFIG 
SPLIT_DIR = "data/split"
BATCH_SIZE = 32
NUM_WORKERS = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CKPT_PATH = "weights/best_classifier_blackout.pth"  # KHÁC 2 bản trước — giữ cả 3 checkpoint để so sánh
Path("weights").mkdir(exist_ok=True)

# LR theo 3 pha
LR_HEAD_ONLY = 1e-3
LR_LAST_BLOCKS = 1e-4
LR_ALL = 1e-5

EPOCHS_P1 = 3   # warm-up head
EPOCHS_P2 = 15  # fine-tune block cuối
EPOCHS_P3 = 5   # full fine-tune (tùy chọn)
PATIENCE = 5

In [ ]:
# DataLoader
train_ds = ChestXrayClassificationDataset(
    f"{SPLIT_DIR}/train", get_train_transforms(), crop_to_lung=True, blackout=True,
)  # KHÁC bản crop: thêm blackout=True
val_ds = ChestXrayClassificationDataset(
    f"{SPLIT_DIR}/val", get_val_transforms(), crop_to_lung=True, blackout=True,
)  # KHÁC bản crop: thêm blackout=True

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)


In [ ]:
model = build_classifier(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)
criterion = nn.CrossEntropyLoss()  # thêm weight=... nếu dữ liệu mất cân bằng
scaler = GradScaler()

def run_epoch(loader, train: bool, optimizer=None):
    model.train() if train else model.eval()
    losses, ys, ps = [], [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in tqdm(loader, leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            if train:
                optimizer.zero_grad()
            with autocast():
                logits = model(x)
                loss = criterion(logits, y)
            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            losses.append(loss.item())
            ys.extend(y.cpu().tolist())
            ps.extend(logits.argmax(1).cpu().tolist())
    macro_f1 = f1_score(ys, ps, average="macro")
    return np.mean(losses), macro_f1


In [ ]:
history = {
    "train_loss": [],
    "val_loss": [],
    "train_f1": [],
    "val_f1": [],
}

In [ ]:
def train_phase(phase_name, epochs, lr, best_f1):
    print(f"\n=== Phase: {phase_name} | LR={lr} | trainable={count_trainable_params(model)} ===")
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    patience_ctr = 0
    for ep in range(epochs):
        tr_loss, tr_f1 = run_epoch(train_loader, train=True, optimizer=optimizer)
        va_loss, va_f1 = run_epoch(val_loader, train=False)
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_f1"].append(tr_f1)
        history["val_f1"].append(va_f1)
        
        scheduler.step()
        print(f"Ep {ep+1:02d} train_loss={tr_loss:.4f} tr_f1={tr_f1:.4f} "
              f"val_loss={va_loss:.4f} val_f1={va_f1:.4f}")

        if va_f1 > best_f1:
            best_f1 = va_f1
            torch.save(model.state_dict(), CKPT_PATH)
            print(f"  Saved best_f1={best_f1:.4f}")
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"  Early stop at epoch {ep+1}")
                break
    return best_f1

best_f1 = 0.0
# Pha 1
freeze_backbone(model)
best_f1 = train_phase("head-only", EPOCHS_P1, LR_HEAD_ONLY, best_f1)

# Pha 2
unfreeze_last_blocks(model, num_blocks=2)
best_f1 = train_phase("last-2-blocks", EPOCHS_P2, LR_LAST_BLOCKS, best_f1)

# Pha 3 (tùy chọn)
unfreeze_all(model)
best_f1 = train_phase("all", EPOCHS_P3, LR_ALL, best_f1)

print(f"\nBEST VAL MACRO F1: {best_f1:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

FIG_DIR = Path("figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. Loss và Macro-F1 theo epoch
# ============================================================
epochs = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history["train_loss"], marker="o", label="Train loss")
axes[0].plot(epochs, history["val_loss"], marker="o", label="Val loss")
axes[0].set_title("Loss theo epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(epochs, history["train_f1"], marker="o", label="Train Macro-F1")
axes[1].plot(epochs, history["val_f1"], marker="o", label="Val Macro-F1")
axes[1].set_title("Macro-F1 theo epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro-F1")
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "loss_f1_curves_blackout.png", dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# 2. Confusion matrix trên validation với best model
# ============================================================
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)

        logits = model(images)
        predictions = logits.argmax(dim=1).cpu().numpy()

        y_pred.extend(predictions)
        y_true.extend(labels.numpy())

class_names = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]
cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.title("Confusion Matrix — Validation Set")
plt.xlabel("Dự đoán")
plt.ylabel("Nhãn thật")
plt.tight_layout()
plt.savefig(FIG_DIR / "confusion_matrix_val_blackout.png", dpi=300, bbox_inches="tight")
plt.show()

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4,
))